# Inventra End-to-End Workflow Notebook

This notebook mirrors the human planning process that the future agents will automate.
It shows how a case moves from a stock check to a recommendation, policy-guided review,
human feedback, revalidation, and finally a safe deterministic purchase request.

In [ ]:
from datetime import datetime
from pprint import pprint

from tools.langchain_tools import build_tool_lookup

agent_tools = build_tool_lookup()
system_tools = build_tool_lookup(include_write_tools=True)

print("Agent-visible tools:")
print(sorted(agent_tools.keys()))
print("\nDeterministic/system tools:")
print([name for name in sorted(system_tools.keys()) if name not in agent_tools])

## 1. Start with a human request

A planner asks one focused question about one product at one warehouse.

In [ ]:
case = {
    "case_id": "CASE-AC003-DEL01",
    "trace_id": "TRACE-AC003-DEL01",
    "sku": "AC-003",
    "warehouse_id": "DEL-01",
    "target_cover_days": 14,
    "selection_strategy": "balanced",
}

print(case)

## 2. Gather product, inventory, and demand evidence

This is what a human planner would check first before deciding whether the case even needs replenishment work.

In [ ]:
product = agent_tools["get_product"].invoke({"sku": case["sku"]})
stock = agent_tools["get_stock_position"].invoke({
    "sku": case["sku"],
    "warehouse_id": case["warehouse_id"],
})
sales = agent_tools["get_sales_velocity"].invoke({
    "sku": case["sku"],
    "warehouse_id": case["warehouse_id"],
    "windows": (7, 30),
})

available_units = stock["on_hand"] - stock["reserved"] + stock["confirmed_inbound"]
daily_velocity = sales["window_7_days"] or sales["window_30_days"]
risk = agent_tools["calculate_stock_risk"].invoke({
    "available_units": available_units,
    "daily_velocity": daily_velocity,
    "target_cover_days": case["target_cover_days"],
    "snapshot_captured_at": stock["captured_at"],
})

print("Product")
pprint(product)
print("\nStock")
pprint(stock)
print("\nSales")
pprint(sales)
print("\nRisk")
pprint(risk)

## 3. Decide whether to stop or continue

A human would stop here if the data is stale or the stock is healthy. Otherwise the case moves into replenishment analysis.

In [ ]:
if risk["stale"]:
    case_status = "BLOCKED"
    decision_reason = "Inventory snapshot is stale"
elif not risk["at_risk"]:
    case_status = "NO_ACTION"
    decision_reason = "Coverage is healthy"
else:
    case_status = "ANALYZE_REPLENISHMENT"
    decision_reason = "Projected cover is below target"

print({"status": case_status, "reason": decision_reason})

## 4. Compare vendor choices

Once risk is confirmed, the planner looks at active offers, vendor reliability, and the cost-versus-speed tradeoff.

In [ ]:
offers = agent_tools["list_vendor_offers"].invoke({"sku": case["sku"]})
performance = agent_tools["get_vendor_performance"].invoke({
    "vendor_ids": [offer["vendor_id"] for offer in offers["offers"]],
})
options = agent_tools["build_vendor_options"].invoke({
    "stock_risk": risk,
    "vendor_offers": offers,
    "vendor_performance": performance,
})
recommendation = agent_tools["recommend_vendor_option"].invoke({
    "case_id": case["case_id"],
    "sku": case["sku"],
    "warehouse_id": case["warehouse_id"],
    "vendor_options": options,
    "strategy": case["selection_strategy"],
})

print("Options summary")
print(f"Eligible options: {len(options['eligible_options'])}")
print("\nCheapest overall")
pprint(options["cheapest_option"])
print("\nFastest overall")
pprint(options["fastest_option"])
print("\nRecommendation")
pprint(recommendation)

## 5. Draft the proposal and review it with policy guidance

Instead of a hard-coded policy function, the agent reads budget evidence and the text policy,
then makes a review recommendation grounded in the evidence.

In [ ]:
budget_month = datetime.fromisoformat(stock["captured_at"]).strftime("%Y-%m")
budget = agent_tools["get_budget_position"].invoke({
    "warehouse_id": case["warehouse_id"],
    "budget_month": budget_month,
})
policy_guidance = agent_tools["get_policy_guidance"].invoke({
    "sku": case["sku"],
    "warehouse_id": case["warehouse_id"],
    "target_cover_days": case["target_cover_days"],
})

proposal = agent_tools["draft_replenishment_proposal"].invoke({
    "case_id": case["case_id"],
    "sku": case["sku"],
    "warehouse_id": case["warehouse_id"],
    "target_cover_days": case["target_cover_days"],
    "stock": stock,
    "sales": sales,
    "risk": risk,
    "budget": budget,
    "recommendation": recommendation,
})
proposal["vendor_evidence_ids"] = [offer["evidence_id"] for offer in offers["offers"]]

policy_concerns = []
if proposal["total_cost"] > budget["remaining"]:
    policy_concerns.append("OVER_BUDGET")
if proposal["expected_arrival"] > proposal["projected_stockout_date"]:
    policy_concerns.append("TIMING_INFEASIBLE")
if risk["stale"]:
    policy_concerns.append("DATA_STALE")
if recommendation["blocked"]:
    policy_concerns.append(recommendation["blocked_reason"])

proposal["policy_passed"] = len(policy_concerns) == 0
proposal["policy_violations"] = policy_concerns
policy_review = {
    "decision": "AWAITING_APPROVAL" if proposal["policy_passed"] else "BLOCKED",
    "concerns": policy_concerns,
    "review_summary": "Agent reviewed evidence against policy.md and found the proposal ready for approval."
    if proposal["policy_passed"]
    else "Agent reviewed evidence against policy.md and found concerns that require revision or escalation.",
    "policy_summary": policy_guidance["summary"],
}

print("Budget")
pprint(budget)
print("\nPolicy guidance summary")
print(policy_guidance["summary"])
print("\nProposal draft")
pprint(proposal)
print("\nPolicy-guided review")
pprint(policy_review)

## 6. Package the approval request

The proposal is now ready to be shown to a named human reviewer.

In [ ]:
approval_packet = agent_tools["prepare_approval_request"].invoke({
    "proposal": proposal,
})

pprint(approval_packet)

## 7. Mimic the human review loop

Humans do not just approve or reject. They often ask for a revision first.
This section shows both a revision request and a final approval.

In [ ]:
revision_review = agent_tools["record_human_review"].invoke({
    "case_id": case["case_id"],
    "proposal": proposal,
    "approver": "Asha Mehta",
    "decision": "REVISE",
    "comments": "Show me the fastest eligible option as well before I approve.",
})

print("Revision request")
pprint(revision_review)

revised_recommendation = agent_tools["recommend_vendor_option"].invoke({
    "case_id": case["case_id"],
    "sku": case["sku"],
    "warehouse_id": case["warehouse_id"],
    "vendor_options": options,
    "strategy": "fastest",
})
revised_proposal = agent_tools["draft_replenishment_proposal"].invoke({
    "case_id": case["case_id"] + "-REV1",
    "sku": case["sku"],
    "warehouse_id": case["warehouse_id"],
    "target_cover_days": case["target_cover_days"],
    "stock": stock,
    "sales": sales,
    "risk": risk,
    "budget": budget,
    "recommendation": revised_recommendation,
})
revised_proposal["vendor_evidence_ids"] = [offer["evidence_id"] for offer in offers["offers"]]
revised_proposal["policy_passed"] = True
revised_proposal["policy_violations"] = []

approval_review = agent_tools["record_human_review"].invoke({
    "case_id": case["case_id"],
    "proposal": revised_proposal,
    "approver": "Asha Mehta",
    "decision": "APPROVED",
    "comments": "Proceed with the expedited option.",
})

print("\nRevised recommendation")
pprint(revised_recommendation)
print("\nRevised proposal")
pprint(revised_proposal)
print("\nApproval review")
pprint(approval_review)

## 8. Revalidate and execute with deterministic tools only

This is the strict boundary. An agent can reach approval, but the final recheck and purchase request creation belong to deterministic system tools.

In [ ]:
revalidation = agent_tools["revalidate_approved_proposal"].invoke({
    "proposal_id": revised_proposal["proposal_id"],
    "proposal_hash": revised_proposal["proposal_hash"],
})

if revalidation["all_checks_pass"]:
    idempotency_key = f"{revised_proposal['proposal_id']}:{approval_review['approver']}"
    purchase_request = system_tools["create_purchase_request"].invoke({
        "proposal": revised_proposal,
        "idempotency_key": idempotency_key,
        "approved_by": approval_review["approver"],
    })
else:
    purchase_request = {"status": "BLOCKED", "reason": revalidation["error_details"]}

pprint(revalidation)
print("\nPurchase request result")
pprint(purchase_request)

## 9. Record an audit event

A human operator should be able to reconstruct what happened from the evidence and review trail.

In [ ]:
audit_event = system_tools["append_audit_event"].invoke({
    "case_id": case["case_id"],
    "trace_id": case["trace_id"],
    "actor": "system",
    "event_type": "purchase_request_created",
    "payload": {
        "proposal_id": revised_proposal["proposal_id"],
        "request_id": purchase_request.get("request_id"),
        "approved_by": approval_review["approver"],
        "recommendation_strategy": revised_recommendation["strategy"],
    },
})

pprint(audit_event)

## 10. Quick failure examples

The same toolset also supports the early-stop cases a human expects to see explained clearly.

In [ ]:
stale_stock = agent_tools["get_stock_position"].invoke({"sku": "AC-002", "warehouse_id": "DEL-01"})
new_sku_sales = agent_tools["get_sales_velocity"].invoke({"sku": "AC-005", "warehouse_id": "DEL-01", "windows": (7, 30)})
unknown_product = agent_tools["get_product"].invoke({"sku": "NONEXISTENT"})

print("Stale inventory example")
pprint(stale_stock)
print("\nInsufficient history example")
pprint(new_sku_sales)
print("\nUnknown product example")
pprint(unknown_product)